# Competición Kaggle


### Sección 1: Carga y EDA

In [ ]:
import pandas as pd
import numpy as np
import re # Para reconocer y extraer patrones dentro de texto( limpieza de tu columna Memory)

# Modelos y utilidades para entrenar y validar
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor
)
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error



In [ ]:
df = pd.read_csv('data/train.csv')
df  

,id,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,697,705,Asus,Chromebook Flip,2 in 1 Convertible,12.5,Full HD / Touchscreen 1920x1080,Intel Core M M3-6Y30 0.9GHz,4GB,64GB Flash Storage,Intel HD Graphics 515,Chrome OS,1.2kg,669.00
1,435,442,Asus,Rog Strix,Gaming,17.3,Full HD 1920x1080,AMD Ryzen 1600 3.2GHz,8GB,256GB SSD + 1TB HDD,AMD Radeon RX 580,Windows 10,3.2kg,1695.00
2,735,743,Lenovo,V310-15IKB (i7-7500U/4GB/1TB/FHD/W10),Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,4GB,1TB HDD,Intel HD Graphics 620,Windows 10,1.85kg,779.00
3,864,875,Dell,XPS 13,Ultrabook,13.3,Quad HD+ / Touchscreen 3200x1800,Intel Core i7 7660U 2.5GHz,16GB,512GB SSD,Intel Iris Plus Graphics 640,Windows 10,1.29kg,2240.00
4,1176,1194,Lenovo,B51-80 (i7-6500U/4GB/1008GB/FHD/W7),Notebook,15.6,Full HD 1920x1080,Intel Core i7 6500U 2.5GHz,4GB,1.0TB Hybrid,Intel HD Graphics 520,Windows 7,2.32kg,825.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
907,920,934,Dell,Vostro 3568,Notebook,15.6,1366x768,Intel Core i5 7200U 2.5GHz,4GB,1TB HDD,AMD Radeon R5 M420,Windows 10,2.18kg,684.99
908,298,303,Lenovo,IdeaPad 310-15ABR,Notebook,15.6,Full HD 1920x1080,AMD A10-Series 9600P 2.4GHz,6GB,1TB HDD,AMD Radeon R5 430,Windows 10,2.4kg,499.00
909,919,933,MSI,GL62M 7RD,Gaming,15.6,Full HD 1920x1080,Intel Core i5 7300HQ 2.5GHz,8GB,128GB SSD + 1TB HDD,Nvidia GeForce GTX 1050,Windows 10,2.2kg,1119.91
910,717,725,Lenovo,110-15ACL (A6-7310/4GB/500GB/W10),Notebook,15.6,1366x768,AMD A6-Series 7310 2GHz,4GB,500GB HDD,AMD Radeon R4,Windows 10,2.19kg,298.00


In [ ]:
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                912 non-null    int64  
 1   laptop_ID         912 non-null    int64  
 2   Company           912 non-null    object 
 3   Product           912 non-null    object 
 4   TypeName          912 non-null    object 
 5   Inches            912 non-null    float64
 6   ScreenResolution  912 non-null    object 
 7   Cpu               912 non-null    object 
 8   Ram               912 non-null    object 
 9   Memory            912 non-null    object 
 10  Gpu               912 non-null    object 
 11  OpSys             912 non-null    object 
 12  Weight            912 non-null    object 
 13  Price_euros       912 non-null    float64
dtypes: float64(2), int64(2), object(10)
memory usage: 99.9+ KB


In [164]:
df.describe()

,id,laptop_ID,Inches,Price_euros
count,912.000000,912.000000,912.000000,912.00000
mean,652.099781,661.273026,15.060746,1126.92034
std,375.428905,380.297415,1.412363,696.08887
min,0.000000,1.000000,10.100000,174.00000
25%,332.500000,338.500000,14.000000,589.00000
50%,655.500000,663.500000,15.600000,952.00000
75%,980.500000,994.500000,15.600000,1499.00000
max,1301.000000,1319.000000,18.400000,4899.00000


In [165]:
print("Valores nulos por columna:\n", df.isnull().sum())


Valores nulos por columna:
 id                  0
laptop_ID           0
Company             0
Product             0
TypeName            0
Inches              0
ScreenResolution    0
Cpu                 0
Ram                 0
Memory              0
Gpu                 0
OpSys               0
Weight              0
Price_euros         0
dtype: int64


### Sección 2: Limpieza y feature‐engineering.

In [ ]:
# 2.1 Limpieza de columnas
df['Ram']    = df['Ram'].str.replace('GB','',regex=False).astype(int) # Quita "GB" y convierte a entero
df['Weight'] = df['Weight'].str.replace('kg','',regex=False).astype(float) # Quita "kg" y convierte a float

# 2.2 Extraer marca de CPU y GPU
df['Cpu_Brand'] = df['Cpu'].str.split().str[0]  # Toma la primera palabra de "Cpu"
df['Gpu_Brand'] = df['Gpu'].str.split().str[0]  # Igual para "Gpu"

# Si no es marca conocida, lo marcamos como "Unknown"
known = ['Intel','AMD','Samsung','Apple','ARM']
df['Cpu_Brand'] = df['Cpu_Brand'].where(df['Cpu_Brand'].isin(known),'Unknown')
df['Gpu_Brand'] = df['Gpu_Brand'].where(df['Gpu_Brand'].isin(known),'Unknown')

# 2.2b Extraer familia de CPU (busca si en el nombre aparece cosas como i3, i5, i7, i9 o Ryzen 3/5/7/9.)
df['Cpu_Family'] = df['Cpu'].str.extract(r'(i[3579]|Ryzen\s*[3579])', expand=False).fillna('Other') # Si no coincide, "Other"

# 2.3 Procesar pantalla
#   - ResX y ResY: resolución horizontal y vertical
df[['ResX','ResY']] = df['ScreenResolution'].str.extract(r'(\d+)x(\d+)', expand=True).astype(int)
#   - PPI: densidad de píxeles
df['PPI'] = np.sqrt(df['ResX']*2 + df['ResY']*2) / df['Inches']
#   - Aspect ratio: relación ancho/alto
df['Aspect'] = df['ResX'] / df['ResY']

# 2.4 Procesar almacenamiento (HDD vs SSD, también Flash)
def split_mem(x):
    """
    Para cada parte de Memory (separada por '+'):
     - saca el número (usando regex)
     - si dice 'TB', lo pasa a GB multiplicando x 1000
     - si dice 'HDD', suma a hdd; si no, a ssd
    Devuelve (hdd, ssd).
    """
    hdd = 0
    ssd = 0
    for part in x.split('+'):
        m = re.search(r'(\d+)', part)       # Busca el número
        if not m:
            continue
        val = int(m.group(1))
        if 'TB' in part.upper():           # TB → GB
            val *= 1000
        if 'HDD' in part.upper():          # Suma al HDD
            hdd += val
        else:
            ssd += val                     # Flash o SSD
    return pd.Series([hdd, ssd])

# Aplica la función y crea columnas
df[['HDD_GB','SSD_GB']] = df['Memory'].apply(split_mem)
df['TotalStore'] = df['HDD_GB'] + df['SSD_GB']  # Almacenamiento total
df['SSD_ratio'] = df['SSD_GB'] / df['TotalStore']  # % de SSD

# 2.5 Índice de portabilidad: peso por pulgada
df['Wt_per_Inch'] = df['Weight'] / df['Inches']


# 2.6 Target encoding (media del precio) por categoría usando KFold
from sklearn.model_selection import KFold

def kfold_target_encoding(df, col, target, n_splits=5, seed=42):
    """
    Para cada fila, asigna la media de 'target' 
    de su categoría en 'col', sin usar su propio valor (KFold).
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = pd.Series(index=df.index, dtype=float) # Out-Of-Fold,  "no usar los mismos datos para calcular y para predecir"
    for train_idx, val_idx in kf.split(df):
        means = df.iloc[train_idx].groupby(col)[target].mean()
        oof.iloc[val_idx] = df.iloc[val_idx][col].map(means)
    # Rellenar categorías nuevas con la media global
    return oof.fillna(df[target].mean())

cat_cols = [
    'Company','TypeName','OpSys',
    'Cpu_Brand','Gpu_Brand','Cpu_Family'
]
for c in cat_cols:
    df[f'{c}_te'] = kfold_target_encoding(df, c, 'Price_euros')

### Sección 3: Preparar x e y

In [167]:
# Lista de columnas para el modelo
numeric_features = [
    'Inches','Ram','Weight',
    'ResX','ResY','PPI','Aspect',
    'HDD_GB','SSD_GB','TotalStore','SSD_ratio',
    'Wt_per_Inch'
]
te_features = [f'{c}_te' for c in cat_cols]

X = df[numeric_features + te_features]
y = np.log1p(df['Price_euros'])  # Target = log(precio)para suavizar la distribución

### Sección 4: StackingRegressor + KFold CV

In [ ]:
# 4.1 Definimos los modelos base
estimators = [
    ('rf', RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1)), # usa todos los procesadores disponibles
    ('gb', GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05, # pequeños pasos de corrección
        max_depth=5,
        random_state=42)),
    ('xgb', XGBRegressor(
        n_estimators=300, 
        learning_rate=0.05,
        max_depth=5,
        objective='reg:squarederror', #  objetivo de regresión (predecir números)
        random_state=42,
        n_jobs=-1))
]

# 4.2 Definimos el meta-modelo(aprende a mezclar salidas de los modelos anteriores)
meta = RidgeCV(alphas=[0.1,1.0,10.0])

# 4.3 Creamos el modelo de stacking
stack = StackingRegressor(
    estimators=estimators, final_estimator=meta,
    passthrough=True, cv=5, n_jobs=-1
)

# 4.4 Validación con KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
neg_mae = cross_val_score(
    stack, X, y, scoring='neg_mean_absolute_error',
    cv=kf, n_jobs=-1
)
mae_scores = -neg_mae
print(f'Stacking CV MAE: {mae_scores.mean():.2f} € ± {mae_scores.std():.2f} €')

# Entrenar con todo
stack.fit(X, y)

Stacking CV MAE: 0.18 € ± 0.01 €


StackingRegressor(cv=5,
                  estimators=[('rf',
                               RandomForestRegressor(max_depth=20,
                                                     min_samples_leaf=3,
                                                     n_estimators=300,
                                                     n_jobs=-1,
                                                     random_state=42)),
                              ('gb',
                               GradientBoostingRegressor(learning_rate=0.05,
                                                         max_depth=5,
                                                         n_estimators=300,
                                                         random_state=42)),
                              ('xgb',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            col...
                                            interaction_constraints=None,
                                            learning_rate=0.05, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=5,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=300, n_jobs=-1,
                                            num_parallel_tree=None, ...))],
                  final_estimator=RidgeCV(alphas=[0.1, 1.0, 10.0]), n_jobs=-1,
                  passthrough=True)

### Sección 5: Predicción final en test.csv y generación de submission.csv

In [169]:
# Leer el test csv
test = pd.read_csv('data/test.csv')

# 5.1 Repetir la misma limpieza que con train 
test['Ram']    = test['Ram'].str.replace('GB','',regex=False).astype(int)
test['Weight'] = test['Weight'].str.replace('kg','',regex=False).astype(float)
test['Cpu_Brand']  = test['Cpu'].str.split().str[0]
test['Gpu_Brand']  = test['Gpu'].str.split().str[0]
test['Cpu_Brand']  = test['Cpu_Brand'].where(test['Cpu_Brand'].isin(known_brands),'Unknown')
test['Gpu_Brand']  = test['Gpu_Brand'].where(test['Gpu_Brand'].isin(known_brands),'Unknown')
test['Cpu_Family'] = test['Cpu'].str.extract(r'(i[3579]|Ryzen\s*[3579])', expand=False).fillna('Other')
test[['ResX','ResY']] = test['ScreenResolution'].str.extract(r'(\d+)x(\d+)', expand=True).astype(int)
test['PPI']    = np.sqrt(test['ResX']*2 + test['ResY']*2) / test['Inches']
test['Aspect'] = test['ResX'] / test['ResY']
test[['HDD_GB','SSD_GB']] = test['Memory'].apply(split_mem)
test['TotalStore'] = test['HDD_GB'] + test['SSD_GB']
test['SSD_ratio']  = test['SSD_GB'] / test['TotalStore']
test['Wt_per_Inch']= test['Weight'] / test['Inches']

# 5.2 Aplicar el target encoding con medias del train
for c in cat_cols:
    test[f'{c}_te'] = test[c].map(df.groupby(c)['Price_euros'].mean()).fillna(df['Price_euros'].mean())

# 5.3 Predecir y deshacer el log
X_test = test[numeric_features + te_features]
y_test_log = stack.predict(X_test)
y_test     = np.expm1(y_test_log)

# 5.4 Crear el csv de submission
submission = pd.DataFrame({'id': test['id'], 'Price_euros': y_test})
submission.to_csv('submission.csv', index=False)
print("¡Submission creado!")

¡Submission creado!
